# Day 074 — Solution: Video Processor

In [ ]:
_VIDEO_SRC = '"""video_processor.py — Day 074: Video Basics.\n\nProcess video files in Python using OpenCV and FFmpeg.\n\nSetup:\n    pip install opencv-python-headless\n    brew install ffmpeg   # macOS\n\nUsage:\n    from video_processor import VideoProcessor\n\n    # Testing — no video file or FFmpeg needed\n    import numpy as np\n    def _mock_info(source):\n        return {\'fps\': 30.0, \'frame_count\': 10, \'width\': 64, \'height\': 64, \'duration_sec\': 0.333}\n    def _mock_capture(source):\n        return [np.zeros((64, 64, 3), dtype=np.uint8) for _ in range(10)]\n\n    proc = VideoProcessor(info_fn=_mock_info, capture_fn=_mock_capture)\n    meta  = proc.info(\'video.mp4\')\n    frames = proc.frames(\'video.mp4\', step=2, max_frames=4)\n    print(meta[\'fps\'], len(frames))   # 30.0  5 (10 frames step=2)\n"""\nimport subprocess\nfrom pathlib import Path\nfrom typing import Callable, Optional\n\n\ndef get_video_info(source, info_fn: Optional[Callable] = None) -> dict:\n    """Return video metadata: fps, frame_count, width, height, duration_sec.\n\n    Args:\n        source:  path to video file (str or Path)\n        info_fn: callable(source) -> dict for testing (no OpenCV needed)\n    """\n    if info_fn is not None:\n        return info_fn(source)\n    import cv2\n    cap = cv2.VideoCapture(str(source))\n    if not cap.isOpened():\n        raise ValueError(f"Cannot open video: {source}")\n    fps         = cap.get(cv2.CAP_PROP_FPS)\n    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n    width       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))\n    height      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\n    cap.release()\n    duration = round(frame_count / fps, 3) if fps > 0 else 0.0\n    return {\n        "fps":         fps,\n        "frame_count": frame_count,\n        "width":       width,\n        "height":      height,\n        "duration_sec": duration,\n    }\n\n\ndef extract_frames(source, step: int = 1, max_frames: Optional[int] = None,\n                   capture_fn: Optional[Callable] = None) -> list:\n    """Extract frames from a video as a list of numpy arrays (BGR, uint8).\n\n    Args:\n        source:     path to video file\n        step:       take every nth frame (1 = every frame, 2 = every other, ...)\n        max_frames: maximum frames to return (None = all)\n        capture_fn: callable(source) -> list[np.ndarray] for testing\n    Returns:\n        list of numpy arrays, shape (H, W, 3), dtype uint8, BGR\n    """\n    if capture_fn is not None:\n        all_frames = capture_fn(source)\n        stepped    = all_frames[::step]\n        return stepped[:max_frames] if max_frames is not None else stepped\n    import cv2\n    cap    = cv2.VideoCapture(str(source))\n    frames = []\n    idx    = 0\n    while True:\n        ret, frame = cap.read()\n        if not ret:\n            break\n        if idx % step == 0:\n            frames.append(frame)\n            if max_frames is not None and len(frames) >= max_frames:\n                break\n        idx += 1\n    cap.release()\n    return frames\n\n\ndef frames_to_video(frames: list, output_path,\n                    fps: float = 30.0, fourcc: str = "mp4v",\n                    writer_fn: Optional[Callable] = None):\n    """Write a list of frames to a video file.\n\n    Args:\n        frames:      list of numpy arrays (H, W, 3) BGR uint8\n        output_path: destination file path\n        fps:         output frame rate\n        fourcc:      four-character codec code (mp4v for MP4, XVID for AVI)\n        writer_fn:   callable(frames, output_path, fps) -> Path for testing\n    Returns:\n        Path to the written video file\n    """\n    if writer_fn is not None:\n        return writer_fn(frames, output_path, fps)\n    import cv2\n    frames = list(frames)\n    if not frames:\n        raise ValueError("frames list is empty")\n    h, w         = frames[0].shape[:2]\n    fourcc_code  = cv2.VideoWriter_fourcc(*fourcc)\n    out_path     = Path(output_path)\n    writer       = cv2.VideoWriter(str(out_path), fourcc_code, fps, (w, h))\n    for frame in frames:\n        writer.write(frame)\n    writer.release()\n    return out_path\n\n\ndef run_ffmpeg(args: list, ffmpeg_fn: Optional[Callable] = None) -> dict:\n    """Run an FFmpeg command and return the result dict.\n\n    Args:\n        args:      FFmpeg arguments (everything after \'ffmpeg -y\')\n        ffmpeg_fn: callable(args) -> dict for testing (no FFmpeg binary needed)\n    Returns:\n        {returncode: int, stdout: str, stderr: str}\n    """\n    if ffmpeg_fn is not None:\n        return ffmpeg_fn(args)\n    result = subprocess.run(\n        ["ffmpeg", "-y"] + list(args),\n        capture_output=True, text=True,\n    )\n    return {\n        "returncode": result.returncode,\n        "stdout":     result.stdout,\n        "stderr":     result.stderr,\n    }\n\n\nclass VideoProcessor:\n    """Process video files using OpenCV and FFmpeg.\n\n    Inject fn parameters for testing without video files or FFmpeg::\n\n        proc = VideoProcessor(\n            info_fn=lambda src: {...},\n            capture_fn=lambda src: [frame1, frame2, ...],\n        )\n    """\n\n    def __init__(self, info_fn: Optional[Callable] = None,\n                 capture_fn: Optional[Callable] = None,\n                 writer_fn: Optional[Callable] = None,\n                 ffmpeg_fn: Optional[Callable] = None) -> None:\n        self._info_fn    = info_fn\n        self._capture_fn = capture_fn\n        self._writer_fn  = writer_fn\n        self._ffmpeg_fn  = ffmpeg_fn\n\n    def info(self, source) -> dict:\n        """Return video metadata dict."""\n        return get_video_info(source, info_fn=self._info_fn)\n\n    def frames(self, source, step: int = 1,\n                max_frames: Optional[int] = None) -> list:\n        """Extract frames as a list of numpy arrays."""\n        return extract_frames(source, step=step, max_frames=max_frames,\n                              capture_fn=self._capture_fn)\n\n    def to_video(self, frames: list, output_path,\n                 fps: float = 30.0, fourcc: str = "mp4v"):\n        """Write frames to a video file. Returns Path."""\n        return frames_to_video(frames, output_path, fps=fps,\n                               fourcc=fourcc, writer_fn=self._writer_fn)\n\n    def run_ffmpeg(self, args: list) -> dict:\n        """Run an FFmpeg command. Returns result dict."""\n        return run_ffmpeg(args, ffmpeg_fn=self._ffmpeg_fn)\n'
from pathlib import Path
Path('video_processor.py').write_text(_VIDEO_SRC, encoding='utf-8')
print('video_processor.py written.')

In [ ]:
import tempfile, numpy as np
from pathlib import Path
from video_processor import (
    get_video_info, extract_frames, frames_to_video, run_ffmpeg, VideoProcessor,
)

_FRAMES = [np.zeros((32, 32, 3), dtype=np.uint8) for _ in range(10)]
_mock_info    = lambda src: {'fps': 30.0, 'frame_count': 10, 'width': 32, 'height': 32, 'duration_sec': 0.333}
_mock_capture = lambda src: _FRAMES
_mock_writer  = lambda frames, path, fps: (Path(path).write_bytes(b'V' * len(frames)), Path(path))[1]
_mock_ffmpeg  = lambda args: {'returncode': 0, 'stdout': '', 'stderr': ''}

# 1. get_video_info
meta = get_video_info('x.mp4', info_fn=_mock_info)
assert set(meta) >= {'fps', 'frame_count', 'width', 'height', 'duration_sec'}
assert isinstance(meta['frame_count'], int)
print("\u2705 get_video_info correct")

# 2. extract_frames
frames = extract_frames('x.mp4', step=2, max_frames=4, capture_fn=_mock_capture)
assert len(frames) <= 4 and frames[0].shape == (32, 32, 3)
print("\u2705 extract_frames correct")

# 3. frames_to_video
with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f:
    tmp = f.name
out = frames_to_video(_FRAMES[:5], tmp, writer_fn=_mock_writer)
assert isinstance(out, Path) and out.exists() and out.stat().st_size > 0
print("\u2705 frames_to_video correct")

# 4. run_ffmpeg
res = run_ffmpeg(['-i', 'in.mp4', 'out.avi'], ffmpeg_fn=_mock_ffmpeg)
assert res['returncode'] == 0 and 'stdout' in res and 'stderr' in res
print("\u2705 run_ffmpeg correct")

# 5. VideoProcessor
proc = VideoProcessor(info_fn=_mock_info, capture_fn=_mock_capture,
                      writer_fn=_mock_writer, ffmpeg_fn=_mock_ffmpeg)
meta2  = proc.info('v.mp4')
frms   = proc.frames('v.mp4', step=3, max_frames=3)
out2   = proc.to_video(frms, tmp)
res2   = proc.run_ffmpeg(['-vn', 'out.mp3'])
assert isinstance(meta2, dict) and len(frms) <= 3
assert isinstance(out2, Path) and res2['returncode'] == 0
print("\u2705 VideoProcessor correct")
print("\nVideo Processor complete!")
